Quick-Commerce Dark Store Catchment & Cold-Chain SLA Diagnostic
## Phase 1: Spatial Catchment Analysis & Data Cleaning (Pandas)

This notebook processes `dark_store_orders_raw.csv` to calculate delivery distance catchment zones (0–1.5km, 1.5–3.0km, 3.0–4.5km, >4.5km), flag 10-minute SLA breaches, evaluate thermal spoilage degradation risks for perishable categories, and model store-level net revenue.

In [1]:
import pandas as pd
import numpy as np

# Task 1: Load and Inspect Raw Order Log
df_raw = pd.read_csv('dark_store_orders_raw.csv')

print("=== RAW ORDER LOG OVERVIEW ===")
print(f"Total Orders: {len(df_raw)}")
print("\nMissing Values Per Column:")
print(df_raw.isnull().sum())
print("\nFirst 5 Rows:")
df_raw.head()

=== RAW ORDER LOG OVERVIEW ===
Total Orders: 1000

Missing Values Per Column:
order_id                 0
store_id                 0
city                     0
zone                     0
category                 0
distance_km              0
sla_target_mins          0
actual_delivery_mins     0
sla_breached             0
fleet_type               0
ambient_temp_c           0
thermal_spoilage_flag    0
order_value_inr          0
spoilage_loss_inr        0
dtype: int64

First 5 Rows:


,order_id,store_id,city,zone,category,distance_km,sla_target_mins,actual_delivery_mins,sla_breached,fleet_type,ambient_temp_c,thermal_spoilage_flag,order_value_inr,spoilage_loss_inr
0,ORD_10001,DS_DEL_02,Delhi-NCR,Noida Sector 62,Beverages,1.42,10,8.6,0,Standard Bike,32.5,0,404.96,0.00
1,ORD_10002,DS_BLR_03,Bengaluru,Whitefield,Ambient Grocery,3.51,15,12.6,0,Standard Bike,37.7,0,1540.29,0.00
2,ORD_10003,DS_DEL_01,Delhi-NCR,Gurugram Cyber City,Fresh Produce,5.46,15,21.4,1,EV Refrig Bike,28.1,0,285.75,0.00
3,ORD_10004,DS_BLR_03,Bengaluru,Whitefield,Frozen Meat,1.20,10,7.7,0,Standard Bike,32.6,0,1467.02,0.00
4,ORD_10005,DS_BLR_03,Bengaluru,Whitefield,Dairy & Ice Cream,5.42,15,23.2,1,Standard Bike,34.8,1,270.56,285.15


## Task 2: Data Hygiene & Standardization
We remove duplicate order records, standardize text columns (store names, product categories), and ensure order value and delivery times are non-negative.


In [5]:
# Drop duplicates
initial_count = len(df_raw)
df_clean = df_raw.drop_duplicates().copy()
print(f"Removed {initial_count - len(df_clean)} duplicate records.")

# Clean text formatting (Using correct column: 'store_id')
df_clean['store_id'] = df_clean['store_id'].astype(str).str.strip().str.title()
df_clean['category'] = df_clean['category'].astype(str).str.strip().str.title()

# Filter out invalid negative times or values (Using correct column: 'actual_delivery_mins')
df_clean = df_clean[(df_clean['actual_delivery_mins'] > 0) & (df_clean['order_value_inr'] > 0)].copy()
print(f"Verified Clean Records: {len(df_clean)} rows")

Removed 0 duplicate records.
Verified Clean Records: 1000 rows


## Task 3: Spatial Catchment Zone Engineering
We assign each order to a catchment distance tier based on the straight-line/route distance from the dark store hub:
- **Inner Zone (0 - 1.5 km)**
- **Mid Zone (1.5 - 3.0 km)**
- **Outer SLA Buffer (3.0 - 4.5 km)**
- **Extreme Radius (> 4.5 km)**

In [6]:
# Bin distances into Catchment Zones
bins = [0, 1.5, 3.0, 4.5, np.inf]
labels = ['0 - 1.5 km (Inner)', '1.5 - 3.0 km (Mid)', '3.0 - 4.5 km (Outer Buffer)', '> 4.5 km (Extreme)']

df_clean['catchment_zone'] = pd.cut(df_clean['distance_km'], bins=bins, labels=labels)

print("Catchment Zone Distribution:")
print(df_clean['catchment_zone'].value_counts())

Catchment Zone Distribution:
catchment_zone
1.5 - 3.0 km (Mid)             297
3.0 - 4.5 km (Outer Buffer)    285
> 4.5 km (Extreme)             221
0 - 1.5 km (Inner)             197
Name: count, dtype: int64


## Task 4: SLA Breach, Thermal Spoilage & Revenue Feature Engineering
- **10-Min SLA Breach:** Triggered if `delivery_time_min > 10`.
- **Thermal Spoilage Flag:** Triggered if perishable item (Dairy/Frozen) + transit time > 15 min or ambient temperature > 32°C.
- **Net Revenue (INR):** `Order Value - Refund Cost - Delivery Cost`.

In [8]:
# 10-Minute SLA Breach Flag (Using correct column: 'actual_delivery_mins')
df_clean['sla_breach_flag'] = (df_clean['actual_delivery_mins'] > 10).astype(int)

# Thermal Degradation/Spoilage Flag for Perishables
perishable_cats = ['Dairy & Milk', 'Frozen Foods', 'Fresh Fruits & Veg', 'Dairy & Ice Cream', 'Fresh Produce']
df_clean['spoilage_flag'] = (
    df_clean['category'].isin(perishable_cats) &
    ((df_clean['actual_delivery_mins'] > 15) | (df_clean['ambient_temp_c'] > 32))
).astype(int)

# Calculate Net Revenue per Order (Assuming a standard 50 INR delivery cost baseline as 'delivery_cost_inr' is not in raw data)
# Using 'spoilage_loss_inr' as the refund/loss cost
delivery_cost_baseline = 50
df_clean['net_revenue_inr'] = df_clean['order_value_inr'] - df_clean['spoilage_loss_inr'] - delivery_cost_baseline

print(f"Total SLA Breaches: {df_clean['sla_breach_flag'].sum()} ({df_clean['sla_breach_flag'].mean()*100:.2f}%)")
print(f"Total Spoilage Incidents: {df_clean['spoilage_flag'].sum()}")

Total SLA Breaches: 725 (72.50%)
Total Spoilage Incidents: 386


## Task 5: Store-Level Performance Summary & File Export
We aggregate KPIs by Dark Store hub and export the processed datasets.

In [11]:
# Aggregating Dark Store Performance
store_summary = df_clean.groupby('store_id').agg(
    total_orders=('order_id', 'count'),
    total_gross_revenue_inr=('order_value_inr', 'sum'),
    total_net_revenue_inr=('net_revenue_inr', 'sum'),
    avg_delivery_time_min=('actual_delivery_mins', 'mean'),
    sla_breach_rate_pct=('sla_breach_flag', lambda x: round(x.mean() * 100, 2)),
    spoilage_count=('spoilage_flag', 'sum')
).reset_index()

print("=== DARK STORE PERFORMANCE SUMMARY ===")
print(store_summary.to_string(index=False))

# Export clean CSVs
df_clean.to_csv('dark_store_orders_clean.csv', index=False)
store_summary.to_csv('dark_store_performance_summary.csv', index=False)
print("\nExported 'dark_store_orders_clean.csv' and 'dark_store_performance_summary.csv' successfully!")

=== DARK STORE PERFORMANCE SUMMARY ===
 store_id  total_orders  total_gross_revenue_inr  total_net_revenue_inr  avg_delivery_time_min  sla_breach_rate_pct  spoilage_count
Ds_Blr_01           136                136792.71              129992.71              13.341176                72.06              61
Ds_Blr_02           121                130672.68              124622.68              13.704959                69.42              51
Ds_Blr_03           111                120487.29              101126.96              13.996396                72.07              38
Ds_Del_01           120                121273.81              115273.81              13.870000                70.83              45
Ds_Del_02           122                123596.05               99868.19              14.041803                72.13              50
Ds_Hyd_01           131                135358.25              128808.25              13.896947                71.76              49
Ds_Mum_01           128              